In [0]:
!pip install kafka-python

In [0]:
dbutils.library.restartPython()

In [0]:
TOPICS = {
    "sales": "sales_stream",
    "regions": "regions_stream",
    "expenses": "expenses_stream",
    "employees": "employees_stream"
}

In [0]:
# Updated regions with Pune and Ahmedabad
REGIONS = [
    (1, "Hyderabad"),
    (2, "Delhi"),
    (3, "Mumbai"),
    (4, "Bengaluru"),
    (5, "Pune"),
    (6, "Ahmedabad")
]

# Regional weights for realistic distribution (metros get more sales)
# Mumbai, Delhi, Bengaluru are top metros, others get less
REGION_WEIGHTS = {
    1: 0.12,  # Hyderabad - 12%
    2: 0.22,  # Delhi - 22% (major metro)
    3: 0.26,  # Mumbai - 26% (largest metro)
    4: 0.24,  # Bengaluru - 24% (IT hub)
    5: 0.10,  # Pune - 10%
    6: 0.06   # Ahmedabad - 6%
}

# Products with REALISTIC prices (reduced by 75% from original)
# This brings sales amounts to realistic levels
PRODUCTS = [
    (1, 1250),    # Product 1: 1250 per unit (was 5000)
    (2, 2000),    # Product 2: 2000 per unit (was 8000)
    (3, 3000),    # Product 3: 3000 per unit (was 12000)
    (4, 3750),    # Product 4: 3750 per unit (was 15000)
    (5, 5000),    # Product 5: 5000 per unit (was 20000)
    (6, 625),     # Product 6: 625 per unit (was 2500)
    (7, 875),     # Product 7: 875 per unit (was 3500)
    (8, 1625),    # Product 8: 1625 per unit (was 6500)
    (9, 2375),    # Product 9: 2375 per unit (was 9500)
    (10, 2750),   # Product 10: 2750 per unit (was 11000)
    (11, 3375),   # Product 11: 3375 per unit (was 13500)
    (12, 4125),   # Product 12: 4125 per unit (was 16500)
    (13, 4500),   # Product 13: 4500 per unit (was 18000)
    (14, 5500),   # Product 14: 5500 per unit (was 22000)
    (15, 6250),   # Product 15: 6250 per unit (was 25000)
    (16, 7000),   # Product 16: 7000 per unit (was 28000)
    (17, 7500),   # Product 17: 7500 per unit (was 30000)
    (18, 8750),   # Product 18: 8750 per unit (was 35000)
    (19, 10000),  # Product 19: 10000 per unit (was 40000)
    (20, 11250),  # Product 20: 11250 per unit (was 45000)
    (21, 1050),   # Product 21: 1050 per unit (was 4200)
    (22, 1950),   # Product 22: 1950 per unit (was 7800)
    (23, 3550),   # Product 23: 3550 per unit (was 14200)
    (24, 8000),   # Product 24: 8000 per unit (was 32000)
    (25, 1000),  # Product 25: 12500 per unit (was 50000)
    (26, 375),    # Product 26: 375 per unit (was 1500)
    (27, 800),    # Product 27: 800 per unit (was 3200)
    (28, 1450),   # Product 28: 1450 per unit (was 5800)
    (29, 1800),   # Product 29: 1800 per unit (was 7200)
    (30, 2450),   # Product 30: 2450 per unit (was 9800)
    (31, 2625),   # Product 31: 2625 per unit (was 10500)
    (32, 3200),   # Product 32: 3200 per unit (was 12800)
    (33, 3875),   # Product 33: 3875 per unit (was 15500)
    (34, 4375),   # Product 34: 4375 per unit (was 17500)
    (35, 4800),   # Product 35: 4800 per unit (was 19200)
    (36, 5375),   # Product 36: 5375 per unit (was 21500)
    (37, 5950),   # Product 37: 5950 per unit (was 23800)
    (38, 6625),   # Product 38: 6625 per unit (was 26500)
    (39, 7300),   # Product 39: 7300 per unit (was 29200)
    (40, 7875),   # Product 40: 7875 per unit (was 31500)
    (41, 8450),   # Product 41: 8450 per unit (was 33800)
    (42, 9125),   # Product 42: 9125 per unit (was 36500)
    (43, 9550),   # Product 43: 9550 per unit (was 38200)
    (44, 10625),  # Product 44: 10625 per unit (was 42500)
    (45, 11950),  # Product 45: 11950 per unit (was 47800)
    (46, 13000),  # Product 46: 13000 per unit (was 52000)
    (47, 13875),  # Product 47: 13875 per unit (was 55500)
    (48, 15000),  # Product 48: 15000 per unit (was 60000)
    (49, 17000),  # Product 49: 17000 per unit (was 68000)
    (50, 18750)   # Product 50: 18750 per unit (was 75000)
]

# Product popularity weights (Pareto principle: 20% products = 80% sales)
PRODUCT_WEIGHTS = (
    [8.0] * 10 +  # Top 10 products (hot sellers)
    [0.5] * 40    # Remaining 40 products (long tail)
)

print(f"Total regions: {len(REGIONS)}")
print(f"Total products: {len(PRODUCTS)}")

In [0]:
ngrokip = "0.tcp.in.ngrok.io:15585"

In [0]:
import json
import time
import uuid
import random
from datetime import datetime, timedelta
from kafka import KafkaProducer

In [0]:
def get_last_id(topic):
    df = spark.sql(f"""
        SELECT COALESCE(MAX(last_id), 0) as last_id
        FROM sales_project.brz.state_store
        WHERE topic = '{topic}'
    """)
    
    result = df.collect()

    if not result:
        return 0

    return int(result[0]["last_id"] or 0)

In [0]:
producer = KafkaProducer(
    bootstrap_servers=ngrokip,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

In [0]:
import uuid
from datetime import datetime, timedelta

def now():
    return datetime.now()

def is_daytime():
    """
    Returns True if current time is between 9 AM and 6 PM (day time)
    Returns False otherwise (night time)
    """
    current_hour = datetime.now().hour
    return 9 <= current_hour < 18

def generate_event_time(late_probability=0.2):
    """
    20% of events will be late
    Late means event_time < ingestion_time
    """
    current = now()

    if random.random() < late_probability:
        delay_minutes = random.randint(5, 120)  # up to 2 hours late
        return current - timedelta(minutes=delay_minutes)

    return current

In [0]:
# Regions are dimensional data - NOT sent via CDC
# They should be inserted manually once and rarely change
# Removing send_region_if_needed() function as regions won't be regenerated

def send_initial_regions_once():
    """
    Call this function ONCE to send all 6 regions to Kafka.
    After that, regions are treated as dimensional data (no CDC).
    """
    for region_id, region_name in REGIONS:
        event_time = generate_event_time(late_probability=0.0)
        
        event = {
            "event_id": str(uuid.uuid4()),
            "region_id": region_id,
            "region_name": region_name,
            "operation": "I",
            "event_time": str(event_time),
            "ingestion_time": str(now())
        }
        
        producer.send(
            TOPICS["regions"],
            key=str(region_id).encode(),
            value=event
        )
    
    producer.flush()
    print(f"Sent all {len(REGIONS)} regions to Kafka (one-time operation)")

In [0]:
# Real Indian employee names pool
INDIAN_FIRST_NAMES = [
    "Rahul", "Priya", "Amit", "Sneha", "Vikram", "Anjali", "Arjun", "Pooja",
    "Rajesh", "Kavita", "Sanjay", "Neha", "Arun", "Divya", "Karthik", "Meera",
    "Suresh", "Lakshmi", "Manoj", "Swati", "Ravi", "Nisha", "Deepak", "Ritu",
    "Anil", "Sunita", "Naveen", "Rekha", "Vishal", "Preeti", "Akash", "Simran",
    "Rohan", "Ananya", "Nikhil", "Shruti", "Ashok", "Geeta", "Sachin", "Vandana",
    "Ramesh", "Shilpa", "Prakash", "Aarti", "Sandeep", "Madhuri", "Varun", "Kriti",
    "Ajay", "Shalini", "Gaurav", "Pallavi", "Harish", "Rani", "Abhishek", "Smita"
]

INDIAN_LAST_NAMES = [
    "Sharma", "Kumar", "Singh", "Patel", "Reddy", "Nair", "Gupta", "Verma",
    "Agarwal", "Joshi", "Rao", "Iyer", "Mehta", "Desai", "Shah", "Malhotra",
    "Khanna", "Chopra", "Bose", "Mukherjee", "Banerjee", "Pillai", "Menon", "Das",
    "Kulkarni", "Naik", "Shetty", "Jain", "Kapoor", "Saxena", "Mishra", "Pandey"
]

def generate_realistic_name():
    """Generate a realistic Indian name"""
    first = random.choice(INDIAN_FIRST_NAMES)
    last = random.choice(INDIAN_LAST_NAMES)
    return f"{first} {last}"

# Role-department mappings
ROLE_DEPARTMENT_MAPPING = {
    "Sales": [
        "Sales Executive",
        "Sales Manager",
        "Sales Associate",
        "Senior Sales Executive"
    ],
    "Support": [
        "Customer Support",
        "Senior Support",
        "Support Associate",
        "Support Manager"
    ],
    "Logistics": [
        "Warehouse Associate",
        "Logistics Manager",
        "Logistics Coordinator",
        "Delivery Manager"
    ],
    "Marketing": [
        "Marketing Specialist",
        "Marketing Manager",
        "Digital Marketing",
        "Brand Manager"
    ]
}

SALARY_RANGES = {
    "associate": (50000, 80000),
    "executive": (70000, 110000),
    "specialist": (80000, 120000),
    "coordinator": (75000, 105000),
    "senior": (110000, 160000),
    "manager": (140000, 200000),
    "lead": (130000, 180000)
}

def get_initial_salary_for_role(role):
    """Returns initial salary based on role seniority"""
    role_lower = role.lower()
    
    if "associate" in role_lower:
        min_sal, max_sal = SALARY_RANGES["associate"]
    elif "manager" in role_lower:
        min_sal, max_sal = SALARY_RANGES["manager"]
    elif "senior" in role_lower or "lead" in role_lower:
        min_sal, max_sal = SALARY_RANGES["senior"]
    elif "coordinator" in role_lower:
        min_sal, max_sal = SALARY_RANGES["coordinator"]
    elif "executive" in role_lower:
        min_sal, max_sal = SALARY_RANGES["executive"]
    elif "specialist" in role_lower:
        min_sal, max_sal = SALARY_RANGES["specialist"]
    else:
        min_sal, max_sal = (60000, 100000)
    
    return random.randint(min_sal, max_sal)

def get_role_for_department(department):
    """Returns a random role that matches the department"""
    dept_normalized = department.capitalize()
    if dept_normalized in ROLE_DEPARTMENT_MAPPING:
        return random.choice(ROLE_DEPARTMENT_MAPPING[dept_normalized])
    return "Employee"

def is_appraisal_season():
    """Returns True if current month is appraisal season (April-May)"""
    month = datetime.now().month
    return month in [4, 5]

def can_get_salary_increment(state, emp_id):
    """Minimum 10 months gap between increments"""
    if "last_salary_increment" not in state:
        state["last_salary_increment"] = {}
    
    if emp_id not in state["last_salary_increment"]:
        return True
    
    last_increment = state["last_salary_increment"][emp_id]
    days_since = (datetime.now() - last_increment).days
    return days_since >= 300

def can_update_employee(state, emp_id):
    """
    Controls when an employee can be updated:
    - After INSERT: Must wait 15-30 days (probation period)
    - After UPDATE: Must wait 10-20 days before next update
    """
    if "last_employee_update" not in state:
        state["last_employee_update"] = {}
    
    if emp_id not in state["last_employee_update"]:
        # Never been updated (just inserted) - requires longer wait
        return True
    
    last_update = state["last_employee_update"][emp_id]
    days_since = (datetime.now() - last_update).days
    
    # Longer gap required: 10-20 days between updates
    min_gap_days = random.randint(10, 20)
    return days_since >= min_gap_days

def mark_employee_inserted(state, emp_id):
    """
    Marks employee as inserted.
    New employees must wait 15-30 days before first update (probation).
    """
    if "last_employee_update" not in state:
        state["last_employee_update"] = {}
    # Mark insert with current timestamp - will need to wait 10-20 days
    state["last_employee_update"][emp_id] = datetime.now()

def mark_employee_updated(state, emp_id):
    """Marks employee as updated"""
    if "last_employee_update" not in state:
        state["last_employee_update"] = {}
    state["last_employee_update"][emp_id] = datetime.now()

def apply_salary_increment(state, emp_id, current_salary, increment_type="appraisal"):
    """Applies salary increment"""
    if increment_type == "promotion":
        hike_percent = random.uniform(0.15, 0.30)
    else:
        hike_percent = random.uniform(0.05, 0.15)
    
    new_salary = int(current_salary * (1 + hike_percent))
    new_salary = min(new_salary, 250000)
    
    if "last_salary_increment" not in state:
        state["last_salary_increment"] = {}
    state["last_salary_increment"][emp_id] = datetime.now()
    
    return new_salary

def get_weighted_region():
    """Returns a region_id with weighted distribution"""
    regions = list(REGION_WEIGHTS.keys())
    weights = list(REGION_WEIGHTS.values())
    return random.choices(regions, weights=weights)[0]

def seed_initial_employees(state, count=20):
    """
    Seed initial employee base on first run.
    Creates 20 employees with good department distribution:
    - 50% Sales (10 employees) - CRITICAL for sales generation
    - 20% Support (4 employees)
    - 20% Logistics (4 employees)
    - 10% Marketing (2 employees)
    """
    if "employees_seeded" in state and state["employees_seeded"]:
        return []
    
    print(f"SEEDING {count} initial employees...")
    
    events = []
    event_time = generate_event_time()
    
    if "all_time_employee_ids" not in state:
        state["all_time_employee_ids"] = set()
    
    departments = (
        ["Sales"] * 10 + 
        ["Support"] * 4 + 
        ["Logistics"] * 4 + 
        ["Marketing"] * 2
    )
    
    for department in departments:
        state["last_employee_id"] += 1
        emp_id = state["last_employee_id"]
        
        emp_name = generate_realistic_name()
        role = get_role_for_department(department)
        salary = get_initial_salary_for_role(role)
        
        after = {
            "employee_name": emp_name,
            "role": role,
            "department": department,
            "region_id": get_weighted_region(),
            "joining_date": str(event_time.date()),
            "salary": salary
        }
        
        state["employees"][str(emp_id)] = after
        state["active_employee_ids"].append(emp_id)
        state["all_time_employee_ids"].add(emp_id)
        
        # Mark as inserted - will need to wait 10-20 days before update
        mark_employee_inserted(state, emp_id)
        
        events.append({
            "event_id": str(uuid.uuid4()),
            "employee_id": emp_id,
            "before": None,
            "after": after,
            "operation": "I",
            "event_time": str(event_time),
            "ingestion_time": str(now())
        })
    
    state["employees_seeded"] = True
    print(f"Seeded {count} employees: {len([e for e in state['employees'].values() if e['department']=='Sales'])} Sales, {len([e for e in state['employees'].values() if e['department']=='Support'])} Support, {len([e for e in state['employees'].values() if e['department']=='Logistics'])} Logistics, {len([e for e in state['employees'].values() if e['department']=='Marketing'])} Marketing")
    
    return events

def generate_employee_events_batch(state):
    """
    Generates 0-5 employee CDC events per batch.
    Operation: U (65%), I (30%), D (5%)
    
    UPDATE INTERVALS:
    - After INSERT: 10-20 days before first update
    - After UPDATE: 10-20 days before next update
    
    IMMUTABLE FIELDS (NEVER change):
    - employee_name
    - joining_date
    - department
    
    MUTABLE FIELDS:
    - region_id, role, salary
    """
    if "all_time_employee_ids" not in state:
        state["all_time_employee_ids"] = set()
        for emp_id in state["employees"].keys():
            state["all_time_employee_ids"].add(int(emp_id))
    
    batch_size = random.randint(0, 4)
    events = []
    employees_in_batch = set()
    
    for _ in range(batch_size):
        op = random.choices(["I", "U", "D"], weights=[0.30, 0.65, 0.05])[0]
        event_time = generate_event_time()
        
        if op == "I" or not state["employees"]:
            # INSERT new employee
            state["last_employee_id"] += 1
            emp_id = state["last_employee_id"]
            
            if emp_id in state["all_time_employee_ids"]:
                continue
            
            emp_name = generate_realistic_name()
            department = random.choice(["Sales", "Support", "Logistics", "Marketing"])
            role = get_role_for_department(department)
            salary = get_initial_salary_for_role(role)
            
            after = {
                "employee_name": emp_name,
                "role": role,
                "department": department,
                "region_id": get_weighted_region(),
                "joining_date": str(event_time.date()),
                "salary": salary
            }
            
            state["employees"][str(emp_id)] = after
            state["active_employee_ids"].append(emp_id)
            state["all_time_employee_ids"].add(emp_id)
            employees_in_batch.add(emp_id)
            
            # Mark as inserted - will need to wait 10-20 days
            mark_employee_inserted(state, emp_id)
            
            events.append({
                "event_id": str(uuid.uuid4()),
                "employee_id": emp_id,
                "before": None,
                "after": after,
                "operation": "I",
                "event_time": str(event_time),
                "ingestion_time": str(now())
            })
        
        else:
            # UPDATE or DELETE
            if not state["employees"]:
                continue
            
            # Filter for eligible employees (10-20 days since last insert/update)
            eligible_employees = [
                int(emp_id) for emp_id in state["employees"].keys()
                if (int(emp_id) not in employees_in_batch and 
                    can_update_employee(state, int(emp_id)))
            ]
            
            if not eligible_employees:
                continue
            
            emp_id = random.choice(eligible_employees)
            employees_in_batch.add(emp_id)
            before = state["employees"][str(emp_id)].copy()
            
            if op == "U":
                # UPDATE
                after = before.copy()
                has_meaningful_change = False
                
                # Preserve immutable fields
                after["employee_name"] = before["employee_name"]
                after["joining_date"] = before["joining_date"]
                after["department"] = before["department"]
                
                # Only 3 types of updates: region transfer, promotion, appraisal
                update_type = random.random()
                
                if update_type < 0.5:
                    # Region transfer (50%)
                    new_region = get_weighted_region()
                    if new_region != before["region_id"]:
                        after["region_id"] = new_region
                        has_meaningful_change = True
                
                elif update_type < 0.75:
                    # Role change/promotion (25%)
                    if can_get_salary_increment(state, emp_id):
                        current_dept = before["department"]
                        new_role = get_role_for_department(current_dept)
                        if new_role != before["role"]:
                            after["role"] = new_role
                            after["salary"] = apply_salary_increment(
                                state, emp_id, before["salary"], "promotion"
                            )
                            has_meaningful_change = True
                
                else:
                    # Annual appraisal (25%)
                    if is_appraisal_season() and can_get_salary_increment(state, emp_id):
                        after["salary"] = apply_salary_increment(
                            state, emp_id, before["salary"], "appraisal"
                        )
                        has_meaningful_change = True
                
                if has_meaningful_change:
                    state["employees"][str(emp_id)] = after
                    mark_employee_updated(state, emp_id)
                    
                    events.append({
                        "event_id": str(uuid.uuid4()),
                        "employee_id": emp_id,
                        "before": before,
                        "after": after,
                        "operation": "U",
                        "event_time": str(event_time),
                        "ingestion_time": str(now())
                    })
            
            else:
                # DELETE
                del state["employees"][str(emp_id)]
                if emp_id in state["active_employee_ids"]:
                    state["active_employee_ids"].remove(emp_id)
                
                mark_employee_updated(state, emp_id)
                
                events.append({
                    "event_id": str(uuid.uuid4()),
                    "employee_id": emp_id,
                    "before": before,
                    "after": None,
                    "operation": "D",
                    "event_time": str(event_time),
                    "ingestion_time": str(now())
                })
    
    return events

In [0]:
def is_weekend():
    return datetime.now().weekday() >= 5

def is_holiday_season():
    """
    Check if current date falls in major Indian festival/sale seasons
    Returns multiplier for holiday spike
    """
    today = datetime.now()
    month = today.month
    day = today.day
    
    # Diwali season (October-November)
    if month == 10 and day >= 15:
        return 2.5
    if month == 11 and day <= 15:
        return 2.5
    
    # New Year Sale (late December to early January)
    if month == 12 and day >= 26:
        return 2.2
    if month == 1 and day <= 5:
        return 2.2
    
    # Republic Day Sale (late January)
    if month == 1 and 23 <= day <= 26:
        return 1.8
    
    # Holi Season (March)
    if month == 3 and 10 <= day <= 15:
        return 1.6
    
    # Independence Day Sale (mid August)
    if month == 8 and 13 <= day <= 15:
        return 2.0
    
    # Amazon/Flipkart Great Indian Festival (September-October)
    if month == 9 and day >= 20:
        return 2.3
    if month == 10 and day <= 10:
        return 2.3
    
    return 1.0

def is_month_end_salary_period():
    """Returns True if today is 1st-5th of month (salary credit period)"""
    day = datetime.now().day
    return 1 <= day <= 5

def is_flash_sale_hour():
    """Random flash sale hours (10% chance)"""
    return random.random() < 0.10

def get_hour_multiplier():
    """
    Returns multiplier based on time of day for B2C patterns
    Peak hours: 10 AM - 4 PM (1.0x)
    Business hours: 9 AM - 6 PM (0.8x)
    Evening: 6 PM - 10 PM (0.9x)
    Night: 10 PM - 9 AM (0.2x)
    """
    hour = datetime.now().hour
    
    if 10 <= hour < 16:
        return 1.0
    elif 9 <= hour < 10 or 16 <= hour < 18:
        return 0.8
    elif 18 <= hour < 22:
        return 0.9
    else:
        return 0.2

def get_sales_employees(state):
    """Returns list of employee IDs from Sales department ONLY"""
    sales_emp_ids = []
    for emp_id in state["active_employee_ids"]:
        emp = state["employees"][str(emp_id)]
        dept = emp.get("department", "").lower()
        if dept == "sales":
            sales_emp_ids.append(emp_id)
    return sales_emp_ids

def get_employee_productivity_multiplier(emp_id, state):
    """
    Sales employees have different productivity levels.
    This multiplier affects BOTH:
    1. How often they're selected for sales (weighted selection)
    2. Their order sizes (quantity boost)
    
    Returns multiplier:
    - 1.5x for top 20% performers (close more deals, selected more often)
    - 1.0x for middle 60% (average performance)
    - 0.6x for bottom 20% (fewer deals, selected less often)
    """
    if "employee_productivity" not in state:
        state["employee_productivity"] = {}
    
    if emp_id not in state["employee_productivity"]:
        rand = random.random()
        if rand < 0.20:
            state["employee_productivity"][emp_id] = 1.5  # Top 20%
        elif rand < 0.80:
            state["employee_productivity"][emp_id] = 1.0  # Middle 60%
        else:
            state["employee_productivity"][emp_id] = 0.6  # Bottom 20%
    
    return state["employee_productivity"][emp_id]

def generate_sales(state):
    """
    Generates realistic sales with natural variance.
    Base: 5-15 sales per batch (NOT fixed buckets)
    Natural growth through multipliers (peak hours, holidays, etc.)
    Result: ~5-15 lakhs per day typically, peaks at 25-30 lakhs during festivals
    
    WEIGHTED DISTRIBUTIONS:
    - Products: Top 10 products = 80% of sales (Pareto)
    - Employees: High performers selected more often, low performers less
    """
    # REALISTIC BASE: 5-15 sales per batch
    base_count = random.randint(5, 10)
    
    # Apply time-of-day multiplier
    time_multiplier = get_hour_multiplier()
    count = int(base_count * time_multiplier)
    
    # Weekend pattern for B2C: HIGHER sales (1.3x)
    if is_weekend():
        count = int(count * 1.3)
    
    # Holiday season spike (natural growth, not fixed bucket)
    holiday_multiplier = is_holiday_season()
    count = int(count * holiday_multiplier)
    
    # Month-end salary period spike
    if is_month_end_salary_period():
        count = int(count * 1.4)
    
    # Flash sale hour (random promotional spikes)
    if is_flash_sale_hour():
        count = int(count * 2.5)
    
    # Minimum activity
    count = max(count, 2)

    records = []
    sales_employee_ids = get_sales_employees(state)
    
    if not sales_employee_ids:
        return records

    # Pre-calculate employee weights for weighted selection
    employee_weights = [get_employee_productivity_multiplier(e, state) for e in sales_employee_ids]

    for _ in range(count):
        state["last_sales_id"] += 1

        # 0.5% null employee_ids for quarantine testing
        if random.random() < 0.005:
            emp_id = None
            region_id = get_weighted_region()
            productivity_multiplier = 1.0
        else:
            # WEIGHTED employee selection: High performers selected MORE often
            emp_id = random.choices(sales_employee_ids, weights=employee_weights)[0]
            emp = state["employees"][str(emp_id)]
            region_id = emp["region_id"]
            productivity_multiplier = get_employee_productivity_multiplier(emp_id, state)

        # Better data quality - only 0.3% null product_ids
        if random.random() < 0.003:
            product_id = None
            unit_price = random.choice([p[1] for p in PRODUCTS])
        else:
            # WEIGHTED product selection: Top 10 products = 80% of sales
            product_id, unit_price = random.choices(PRODUCTS, weights=PRODUCT_WEIGHTS)[0]

        # Realistic quantity distribution
        rand = random.random()
        if rand < 0.6:
            quantity = random.randint(1, 2)
        elif rand < 0.9:
            quantity = random.randint(3, 5)
        else:
            quantity = random.randint(6, 15)
        
        # High-performing employees close slightly larger orders
        if productivity_multiplier > 1.0 and random.random() < 0.3:
            quantity = int(quantity * 1.2)

        sales_amount = unit_price * quantity

        # Returns/refunds are rare - only 0.5%
        if random.random() < 0.005:
            sales_amount = -abs(sales_amount)

        event_time = generate_event_time()

        record = {
            "event_id": str(uuid.uuid4()),
            "sales_id": state["last_sales_id"],
            "employee_id": emp_id,
            "region_id": region_id,
            "product_id": product_id,
            "quantity": quantity,
            "sales_amount": sales_amount,
            "event_time": str(event_time),
            "ingestion_time": str(now())
        }

        records.append(record)

    return records

In [0]:
def is_payroll_day():
    """Returns True if today is the 30th (monthly payroll day)"""
    day = datetime.now().day
    return day == 30

def get_logistics_seasonal_multiplier():
    """Returns multiplier for logistics costs during peak seasons"""
    today = datetime.now()
    month = today.month
    day = today.day
    
    if month == 10 and day >= 15:
        return 2.2
    if month == 11 and day <= 15:
        return 2.2
    
    if month == 12 and day >= 26:
        return 1.9
    if month == 1 and day <= 5:
        return 1.9
    
    if month == 9 and day >= 20:
        return 2.0
    if month == 10 and day <= 10:
        return 2.0
    
    if month == 8 and 13 <= day <= 15:
        return 1.6
    
    return 1.0

def generate_expenses(state):
    """
    Generates realistic expenses proportional to sales.
    
    CRITICAL: On 30th of every month (payroll day):
    - ALL active employees receive salary expenses
    - Each employee = 1 salary expense record
    
    Regular days: 3-8 expenses per batch (other expense types)
    """
    records = []
    event_time = generate_event_time()
    
    # PAYROLL DAY (30th): Pay ALL active employees
    if is_payroll_day():
        print(f"PAYROLL DAY: Generating salary expenses for {len(state['active_employee_ids'])} active employees")
        
        for emp_id in state['active_employee_ids']:
            state["last_expense_id"] += 1
            
            employee = state["employees"][str(emp_id)]
            region_id = employee["region_id"]
            salary = employee.get("salary", 80000)  # Default if salary missing
            
            record = {
                "event_id": str(uuid.uuid4()),
                "expense_id": state["last_expense_id"],
                "employee_id": emp_id,
                "region_id": region_id,
                "expense_type": random.choice(["Salary", "salary", "SALARY"]),
                "expense_amount": salary,
                "event_time": str(event_time),
                "ingestion_time": str(now())
            }
            
            records.append(record)
        
        return records
    
    # REGULAR DAYS: Generate other expense types
    base_count = random.randint(2, 5)
    
    if is_daytime():
        expense_count = base_count
    else:
        expense_count = int(base_count * 0.5)
    
    if is_weekend():
        expense_count = int(expense_count * 0.9)
    
    for _ in range(expense_count):
        state["last_expense_id"] += 1

        # Most expenses tied to employees
        if state["active_employee_ids"] and random.random() > 0.1:
            emp_id = random.choice(state["active_employee_ids"])
            employee = state["employees"][str(emp_id)]
            region_id = employee["region_id"]
            emp_department = employee.get("department", "").lower()
        else:
            emp_id = None
            region_id = get_weighted_region()
            emp_department = None

        if random.random() < 0.003:
            emp_id = None

        # Expense type distribution (NO salaries on regular days)
        expense_type_options = [
            ("Commission", "commission", "COMMISSION", "Bonus"),
            ("Marketing", "marketing", "MARKETING"),
            ("Infrastructure", "Infra", "INFRASTRUCTURE"),
            ("Customer Support", "Support", "CUSTOMER SUPPORT"),
            ("Logistics", "logistics", "LOGISTICS"),
            ("Operations", "operations", "OPERATIONS"),
            ("Training", "training", "TRAINING")
        ]
        
        type_weights = [0.15, 0.30, 0.20, 0.15, 0.15, 0.10, 0.05]
        
        expense_type_group = random.choices(expense_type_options, weights=type_weights)[0]
        expense_type = random.choice(expense_type_group)

        # REALISTIC expense amounts
        if expense_type.lower() in ["commission", "bonus"]:
            if emp_department == "sales":
                expense_amount = random.randint(8000, 40000)
            elif emp_department == "support":
                expense_amount = random.randint(4000, 16000)
            else:
                expense_amount = random.randint(2000, 8000)
        
        elif expense_type.lower() == "marketing":
            expense_amount = random.randint(6000, 100000)
        
        elif expense_type.lower() in ["infra", "infrastructure"]:
            expense_amount = random.randint(12000, 120000)
        
        elif expense_type.lower() in ["support", "customer support"]:
            expense_amount = random.randint(4000, 32000)
        
        elif expense_type.lower() == "logistics":
            base_logistics = random.randint(8000, 80000)
            seasonal_multiplier = get_logistics_seasonal_multiplier()
            expense_amount = int(base_logistics * seasonal_multiplier)
        
        elif expense_type.lower() == "operations":
            expense_amount = random.randint(3200, 32000)
        
        else:  # training
            expense_amount = random.randint(1200, 12000)

        if random.random() < 0.002:
            expense_amount = -abs(expense_amount)

        record = {
            "event_id": str(uuid.uuid4()),
            "expense_id": state["last_expense_id"],
            "employee_id": emp_id,
            "region_id": region_id,
            "expense_type": expense_type,
            "expense_amount": expense_amount,
            "event_time": str(event_time),
            "ingestion_time": str(now())
        }

        records.append(record)

    return records

In [0]:
# from kafka.admin import KafkaAdminClient, NewTopic

# for topic_key, topic_name in TOPICS.items():
#     print(topic_key, topic_name)

#     admin_client = KafkaAdminClient(bootstrap_servers=ngrokip)
#     try:
#         new_topic = NewTopic(name=topic_name, num_partitions=1, replication_factor=1)
#         admin_client.create_topics([new_topic])
#         print(f"Created topic: {topic_name}")
#     except Exception as e:
#         print(f"Topic creation error or already exists: {e}")

In [0]:
import json
import os
from pyspark.sql import Row

def load_state():
    """Load state from state_store table."""
    return {
        "last_sales_id": get_last_id("sales"),
        "last_expense_id": get_last_id("expenses"),
        "last_employee_id": get_last_id("employees"),  # Track employee_id state
        "employees": {},
        "active_employee_ids": [],
        "all_time_employee_ids": set()
    }

state = load_state()
print(f"State loaded: sales_id={state['last_sales_id']}, expense_id={state['last_expense_id']}, employee_id={state['last_employee_id']}")

def persist_state_to_table(state):
    """Persist current state to state_store table"""
    spark.sql(f"""
        MERGE INTO sales_project.brz.state_store t
        USING (
            SELECT 'sales' as topic, {state['last_sales_id']} as last_id, current_timestamp() as updated_at
            UNION ALL
            SELECT 'expenses' as topic, {state['last_expense_id']} as last_id, current_timestamp() as updated_at
            UNION ALL
            SELECT 'employees' as topic, {state['last_employee_id']} as last_id, current_timestamp() as updated_at
        ) s
        ON t.topic = s.topic
        WHEN MATCHED THEN UPDATE SET 
            t.last_id = s.last_id,
            t.updated_at = s.updated_at
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
# IMPORTANT: Send all regions ONCE before starting the main loop
# Uncomment the line below on first run to send all 6 regions to Kafka
# send_initial_regions_once()

batch = 0

# SEED initial employees ONLY if last_employee_id < 20 (first run ever)
if state["last_employee_id"] < 10:
    print("First run detected - seeding initial employees...")
    seed_events = seed_initial_employees(state, count=20)
    for e in seed_events:
        producer.send(
            TOPICS["employees"],
            key=str(e["employee_id"]).encode(),
            value=e
        )
    producer.flush()
    persist_state_to_table(state)
    print(f"Seeded {len(seed_events)} initial employees\n")
else:
    print(f"Resuming from employee_id={state['last_employee_id']} (employees already seeded)\n")

while batch < 10:

    # Generate 0-5 employee CDC events per batch (U=65%, I=30%, D=5%)
    emp_events = generate_employee_events_batch(state)
    for e in emp_events:
        producer.send(
            TOPICS["employees"],
            key=str(e["employee_id"]).encode(),
            value=e
        )

    # Generate realistic sales (5-15 base per batch, natural growth via multipliers)
    sales_events = generate_sales(state)
    for s in sales_events:
        producer.send(
            TOPICS["sales"],
            key=str(s["sales_id"]).encode(),
            value=s
        )

    # Generate realistic expenses (3-8 base per batch, proportional to sales)
    expense_events = generate_expenses(state)
    for ex in expense_events:
        producer.send(
            TOPICS["expenses"],
            key=str(ex["expense_id"]).encode(),
            value=ex
        )

    producer.flush()
    persist_state_to_table(state)

    print(f"""
    Batch {batch + 1}:
    Employees: {len(emp_events)}
    Sales: {len(sales_events)}
    Expenses: {len(expense_events)}
    """)

    print(f"Sales ID: {state['last_sales_id']}, Expense ID: {state['last_expense_id']}, Employee ID: {state['last_employee_id']}")
    print(f"Active employees: {len(state['active_employee_ids'])} (Sales: {len([e for e in state['employees'].values() if e['department']=='Sales'])}\n")

    batch += 1
    time.sleep(2)

print("\n=== Producer completed ===")
print(f"Total sales: {state['last_sales_id']}")
print(f"Total expenses: {state['last_expense_id']}")
print(f"Total employees: {state['last_employee_id']}")
print(f"Active employees: {len(state['active_employee_ids'])}")
print(f"Sales employees: {len([e for e in state['employees'].values() if e['department']=='Sales'])}")

In [0]:
%sql
select * from sales_project.brz.state_store;